In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path
import pandas as pd
import requests
from ift6758.llm.get_better import (
    extract_text,
    load_function,
    format_test_report,
    build_refine_prompt,
    all_passed,
)
from ift6758.llm.chat import confirm, ask_user
from ift6758.llm.prompts import PROFILES, OG_TACHE_1
from ift6758.llm.qwen_log import create_log, log_call
from ift6758.llm.qwen_utils import (
    load_qwen,
    build_messages,
    generate,
    ask_qwen,
    check_env,
)
from ift6758.llm.rag import (
    load_doc,
    extract_section,
    split_doc,
    load_embedder,
    embed,
    retrieve,
    format_context,
    DOCS_DIR,
)
from ift6758.llm.test_both_code import run_tests, run_season_tests

check_env()  # Demo 2, for reproductibility

Objectif: Créer une plus petite version du json (car il est énorme, trop gros pour un bon contexte pour le petit modèle). Ca remplace le RAG de la question 1.

plays = [55(tir),87(but),1001(but filet désert)] if p["evenId"] in game
homeTeam, awayTeam
rosterSpots = nom de marqueur, tireur, gardiens
context = "Sample of one real game JSON (json number), structure is identical for all game." + json.dumps(extrait, indent=2)
extra = "context_source":"Game sample num_game, events 55/87/1001

Je me suis fait avoir. Le joeur change de nom selon l'event:Un tir a details.shootingPlayerId, mais un but a details.scoringPlayerId (pas de shootingPlayerId). Ensuite, s'il y a un but désert, le but n'a pas de clé goalieInNetId.

In [ ]:
tidy_task = """
Write a Python function json2tidy_llm(raw_data).
Goal: Convert one NHL play-by-play game (JSON) into a tidy pandas DataFrame, one row per event

Input:
- json2tidy_llm(raw_data) accepts either a str (path to a game JSON file, read as UTF-8) or a dict (one game, already loaded).
- The context shows a sample of the real JSON structure. All games have this structure.

Definitions (from the NHL play-by-play JSON):
- Game level: id = game ID, season (e.g. 20222023), gameType (2 = regular season, 3 = playoffs), homeTeam.id, awayTeam.id.
- Events are in plays. Keep only plays whose typeDescKey is "shot-on-goal" or "goal".
- The shooter is details.shootingPlayerId for a shot, details.scoringPlayerId for a goal.
- Player names are in rosterSpots: playerId -> firstName["default"] + " " + lastName["default"].
- details.goalieInNetId is absent when the net is empty.
- situationCode has 4 digits: away goalie, away skaters, home skaters, home goalie. Example: "1451" = away team has 4 skaters, home team has 5.

Output columns (strictly use these names):
- game_id, event_id, season, game_type
- period (periodDescriptor.number), time (timeInPeriod, "MM:SS")
- team_id (details.eventOwnerTeamId, the team that took the shot)
- event_type ("shot-on-goal" or "goal")
- x_coord, y_coord
- shooter (full name), goalie (full name of goalieInNetId)
- shot_type (details.shotType)
- empty_net (True if goalieInNetId is absent, else False)
- strength: for goals only, compare the skaters of the scoring team with the skaters of the other team:
    more = "power play", fewer = "shorthanded", equal = "even". For shots, strength is missing.

Requirements:
- Use pandas and json. Include imports and a docstring.
- One row per event: (game_id, event_id) must be unique.
- Keep unavailable values as missing (NaN or None). Never replace them with 0, False or "".
- Return a pandas DataFrame.
Return only json2tidy_llm() and its imports in a Python code block.
"""